# ML_Fundamentals Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Rules versus learning.** Hand-written rules fire on keywords alone — and misfire on innocent sentences, which is exactly why we let data teach the pattern.

In [ ]:
def rule_spam(text):
    """Flag SPAM when at least two suspicious keywords appear."""
    spam_words = ["free", "winner", "prize"]
    hits = sum(1 for word in spam_words if word in text.lower())
    return "SPAM" if hits >= 2 else "HAM"

emails = [
    "winner winner claim your free prize",               # truly spam
    "our design tool won first prize at the conference", # ham - trips one word
]
for mail in emails:
    print(rule_spam(mail), "<-", mail)

# Keyword rules are brittle: innocent mail trips them, and spammers simply
# dodge the known words. Learning the pattern from examples scales better.

**2. Meet iris.** A Bunch becomes a tidy DataFrame in two lines: `.data` as the body, `.feature_names` as headers, `.target` attached as a column.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species_code"] = iris.target

print("Shape:", df.shape)                      # (150, 5)
print("Species:", iris.target_names.tolist())

**3. X marks the features.** By convention `X` is a 2-D feature matrix (rows = examples) and `y` a 1-D label vector — one label per row.

In [ ]:
X = df[iris.feature_names]
y = df["species_code"]

print("X shape:", X.shape)    # (150, 4) - rows are flowers, columns are features
print("y shape:", y.shape)    # (150,)   - one label per flower
print("First example:", X.iloc[0].to_dict(), "-> label", y.iloc[0])

## Part 2 — Practice

**4. Regression or classification.** A continuous numeric target makes it supervised REGRESSION; bucketing that number into categories would flip it to classification.

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
df_d = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df_d["target"] = diabetes.target

print("Shape:", df_d.shape)
print("Features:", df_d.columns.tolist())
print(df_d["target"].describe().round(1))

# The target is a continuous progression score (25 .. 346), not a category
# -> supervised REGRESSION. Cutting scores into low/high risk bands would
# turn the very same data into classification.

**5. Sort the paradigms.** Number targets = regression, category targets = classification, no labels = unsupervised, reward feedback = reinforcement.

In [ ]:
paradigms = {
    "a) tomorrow's temperature":      "supervised-regression",
    "b) customer groups, no labels":  "unsupervised",
    "c) spam vs ham, labelled mails": "supervised-classification",
    "d) game agent from rewards":     "reinforcement",
    "e) flat price estimate":         "supervised-regression",
}
for scenario, kind in paradigms.items():
    print(f"{scenario:<32} -> {kind}")

**6. Inside the Bunch.** Every loader hands over the same quartet: numeric matrix, label vector, column names, class names.

In [ ]:
print("Type:", type(iris).__name__)
print(".data           ->", iris.data.shape)
print(".target         ->", iris.target[:5], "...")
print(".feature_names  ->", iris.feature_names)
print(".target_names   ->", iris.target_names.tolist())

**7. The module in miniature.** Train score = open-book exam (memorisation), test score = closed-book exam (generalisation) — only the second pays rent.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)

print(f"train accuracy: {knn.score(X_tr, y_tr):.3f}   <- open-book exam")
print(f"test  accuracy: {knn.score(X_te, y_te):.3f}   <- closed-book exam")

# The TEST score is the honest one: those rows took no part in fitting,
# so they measure generalisation instead of memorisation.

## Part 3 — Challenge

**8. Baseline before bragging.** Guessing the most frequent species already scores ~0.33 here; any model worth deploying must clear that bar.

In [ ]:
baseline = pd.Series(y_te).value_counts(normalize=True).max()
knn_acc = knn.score(X_te, y_te)

print(f"majority-class baseline : {baseline:.3f}")
print(f"KNN test accuracy       : {knn_acc:.3f}")
print("Beats the baseline?", knn_acc > baseline)

**9. Who generalises better.** The unlimited tree grows a branch per training row — flawless at home, shakier away; the train−test gap IS the memorisation meter.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

contenders = {
    "DecisionTree (unlimited)": DecisionTreeClassifier(random_state=42),
    "LogisticRegression":       LogisticRegression(max_iter=5000),
}
for name, model in contenders.items():
    model.fit(X_tr, y_tr)
    print(f"{name:<26} train {model.score(X_tr, y_tr):.3f}"
          f" | test {model.score(X_te, y_te):.3f}")

# The unlimited tree owns the bigger gap: near-perfect train accuracy bought
# by memorising individual rows, paid for on unseen flowers. Lesson 07
# dissects this trade-off in full.